In [13]:
# Phase 2: Feature Engineering

# Load the preprocessed data and create new features for modeling.

In [14]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Adjust display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries imported.")

Libraries imported.


In [15]:
# Define path for processed data
data_dir = '../data'
input_file = os.path.join(data_dir, 'flights_processed_v1.csv')

if not os.path.exists(input_file):
    raise FileNotFoundError(f"Processed data file not found at {input_file}. Make sure 02_data_preprocessing.ipynb ran successfully.")

print(f"Loading processed data from: {input_file}")

df = pd.read_csv(input_file, parse_dates=['FL_DATE']) # Parse FL_DATE immediately

print("Processed dataset loaded successfully.")
print(f"Shape: {df.shape}")
print("Data types:")
df.info()

display(df.head())

print("DataFrame columns immediately after loading:", df.columns.tolist())

Loading processed data from: ../data\flights_processed_v1.csv
Processed dataset loaded successfully.
Shape: (2913802, 9)
Data types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2913802 entries, 0 to 2913801
Data columns (total 9 columns):
 #   Column            Dtype         
---  ------            -----         
 0   FL_DATE           datetime64[ns]
 1   AIRLINE           object        
 2   ORIGIN            object        
 3   DEST              object        
 4   DISTANCE          float64       
 5   CRS_DEP_TIME      int64         
 6   CRS_ARR_TIME      int64         
 7   CRS_ELAPSED_TIME  float64       
 8   IS_DELAYED        int64         
dtypes: datetime64[ns](1), float64(2), int64(3), object(3)
memory usage: 200.1+ MB


,FL_DATE,AIRLINE,ORIGIN,DEST,DISTANCE,CRS_DEP_TIME,CRS_ARR_TIME,CRS_ELAPSED_TIME,IS_DELAYED
0,2019-01-09,United Air Lines Inc.,FLL,EWR,1065.0,1155,1501,186.0,0
1,2022-11-19,Delta Air Lines Inc.,MSP,SEA,1399.0,2120,2315,235.0,0
2,2022-07-22,United Air Lines Inc.,DEN,MSP,680.0,954,1252,118.0,0
3,2023-03-06,Delta Air Lines Inc.,MSP,SFO,1589.0,1609,1829,260.0,1
4,2020-02-23,Spirit Air Lines,MCO,DFW,985.0,1840,2041,181.0,0


DataFrame columns immediately after loading: ['FL_DATE', 'AIRLINE', 'ORIGIN', 'DEST', 'DISTANCE', 'CRS_DEP_TIME', 'CRS_ARR_TIME', 'CRS_ELAPSED_TIME', 'IS_DELAYED']


In [16]:
print("Creating date-based features...")

# Ensure FL_DATE is datetime
if not pd.api.types.is_datetime64_any_dtype(df['FL_DATE']):
    print("Converting FL_DATE to datetime again...")
    df['FL_DATE'] = pd.to_datetime(df['FL_DATE'])

# Extract date components
df['YEAR'] = df['FL_DATE'].dt.year
df['MONTH'] = df['FL_DATE'].dt.month
df['DAY'] = df['FL_DATE'].dt.day # Day of month
df['DAY_OF_WEEK'] = df['FL_DATE'].dt.dayofweek # Monday=0, Sunday=6
df['DAY_OF_YEAR'] = df['FL_DATE'].dt.dayofyear
df['WEEK_OF_YEAR'] = df['FL_DATE'].dt.isocalendar().week.astype(int) # Use isocalendar for week

print("Date features created: YEAR, MONTH, DAY, DAY_OF_WEEK, DAY_OF_YEAR, WEEK_OF_YEAR")
display(df[['FL_DATE', 'YEAR', 'MONTH', 'DAY', 'DAY_OF_WEEK', 'DAY_OF_YEAR', 'WEEK_OF_YEAR']].head())

Creating date-based features...
Date features created: YEAR, MONTH, DAY, DAY_OF_WEEK, DAY_OF_YEAR, WEEK_OF_YEAR


,FL_DATE,YEAR,MONTH,DAY,DAY_OF_WEEK,DAY_OF_YEAR,WEEK_OF_YEAR
0,2019-01-09,2019,1,9,2,9,2
1,2022-11-19,2022,11,19,5,323,46
2,2022-07-22,2022,7,22,4,203,29
3,2023-03-06,2023,3,6,0,65,10
4,2020-02-23,2020,2,23,6,54,8


In [17]:
print("\nCreating time-based features from CRS_DEP_TIME...")

# Check the format and type of CRS_DEP_TIME
print(f"CRS_DEP_TIME type: {df['CRS_DEP_TIME'].dtype}")
print(f"Sample CRS_DEP_TIME values: {df['CRS_DEP_TIME'].head().tolist()}")

# Pad the time with leading zeros to ensure HHMM format (e.g., 730 -> 0730, 5 -> 0005)
# Convert to string first
df['CRS_DEP_TIME_STR'] = df['CRS_DEP_TIME'].astype(int).astype(str).str.zfill(4)

# Handle potential invalid times like '2400' which might appear in raw data
# Convert valid times to datetime objects, coercing errors to NaT (Not a Time)
# Format '%H%M' assumes HHMM format
temp_time = pd.to_datetime(df['CRS_DEP_TIME_STR'], format='%H%M', errors='coerce')

# Check for any parsing errors (NaT values)
nat_count = temp_time.isnull().sum()
if nat_count > 0:
    print(f"Warning: Found {nat_count} invalid CRS_DEP_TIME values that couldn't be parsed (e.g., >= 2400). These will result in NaN time features.")
    # Optional: Inspect the rows causing issues
    # display(df[temp_time.isnull()]['CRS_DEP_TIME_STR'].value_counts())

# Extract Hour and Minute
df['DEP_HOUR'] = temp_time.dt.hour
df['DEP_MINUTE'] = temp_time.dt.minute # Optional, Hour is usually more important

# Create Time-of-Day Bins (example)
# Adjust bins as needed based on domain knowledge or analysis
bins = [-1, 5, 11, 16, 20, 24] # Bins: Night (0-5), Morning (6-11), Afternoon (12-16), Evening (17-20), Late Night (21-23)
labels = ['Night', 'Morning', 'Afternoon', 'Evening', 'Late_Night']
df['DEP_TIME_CATEGORY'] = pd.cut(df['DEP_HOUR'], bins=bins, labels=labels, right=True)

# Handle any NaNs created by invalid times or binning issues if necessary
# For now, let's check if any NaNs exist in the new features
print(f"NaNs in DEP_HOUR: {df['DEP_HOUR'].isnull().sum()}")
print(f"NaNs in DEP_MINUTE: {df['DEP_MINUTE'].isnull().sum()}")
print(f"NaNs in DEP_TIME_CATEGORY: {df['DEP_TIME_CATEGORY'].isnull().sum()}")
# Simple strategy: fill NaNs if any (e.g., with mode or a specific category)
if df['DEP_HOUR'].isnull().any():
    # Example: fill with mode hour, or 0
    mode_hour = df['DEP_HOUR'].mode()[0]
    print(f"Filling NaN DEP_HOUR with mode: {mode_hour}")
    df['DEP_HOUR'].fillna(mode_hour, inplace=True)
    df['DEP_MINUTE'].fillna(0, inplace=True) # Fill minute with 0
    # Recalculate category if NaNs were filled
    df['DEP_TIME_CATEGORY'] = pd.cut(df['DEP_HOUR'], bins=bins, labels=labels, right=True)


print("Time features created: DEP_HOUR, DEP_MINUTE, DEP_TIME_CATEGORY")
display(df[['CRS_DEP_TIME', 'DEP_HOUR', 'DEP_MINUTE', 'DEP_TIME_CATEGORY']].head())

# Drop the intermediate string column
df = df.drop(columns=['CRS_DEP_TIME_STR'])


Creating time-based features from CRS_DEP_TIME...
CRS_DEP_TIME type: int64
Sample CRS_DEP_TIME values: [1155, 2120, 954, 1609, 1840]
NaNs in DEP_HOUR: 0
NaNs in DEP_MINUTE: 0
NaNs in DEP_TIME_CATEGORY: 0
Time features created: DEP_HOUR, DEP_MINUTE, DEP_TIME_CATEGORY


,CRS_DEP_TIME,DEP_HOUR,DEP_MINUTE,DEP_TIME_CATEGORY
0,1155,11,55,Morning
1,2120,21,20,Late_Night
2,954,9,54,Morning
3,1609,16,9,Afternoon
4,1840,18,40,Evening


In [18]:
# --- Start Replacement for Cell 6 ---
print("Columns PRESENT AT START of Cell 6:", df.columns.tolist())

print("Identifying intended categorical features and checking cardinality...")

# Define columns we INTEND to treat as categorical
intended_categorical = ['AIRLINE', 'ORIGIN', 'DEST', 'DEP_TIME_CATEGORY']

# Check which of these actually exist in the DataFrame
existing_intended_categorical = [col for col in intended_categorical if col in df.columns]
print(f"Found intended categorical columns in df: {existing_intended_categorical}")

# --- Ensure correct data types BEFORE encoding ---
# Convert columns intended for OHE or that need specific types for checks
if 'AIRLINE' in existing_intended_categorical:
    if df['AIRLINE'].dtype != 'object':
        print(f"Converting AIRLINE from {df['AIRLINE'].dtype} to string/object.")
        df['AIRLINE'] = df['AIRLINE'].astype(str) # Force to string/object type

if 'DEP_TIME_CATEGORY' in existing_intended_categorical:
    if df['DEP_TIME_CATEGORY'].dtype.name != 'category':
         # Ensure it's treated as object/category for get_dummies
         print(f"Converting DEP_TIME_CATEGORY from {df['DEP_TIME_CATEGORY'].dtype} to string/object.")
         df['DEP_TIME_CATEGORY'] = df['DEP_TIME_CATEGORY'].astype(str)

# ORIGIN/DEST should be object type for frequency encoding mapping
if 'ORIGIN' in existing_intended_categorical and df['ORIGIN'].dtype != 'object':
     print(f"Converting ORIGIN from {df['ORIGIN'].dtype} to string/object.")
     df['ORIGIN'] = df['ORIGIN'].astype(str)
if 'DEST' in existing_intended_categorical and df['DEST'].dtype != 'object':
     print(f"Converting DEST from {df['DEST'].dtype} to string/object.")
     df['DEST'] = df['DEST'].astype(str)
# --- End Type Conversion ---


# Calculate cardinality FOR THE INTENDED columns that exist
if existing_intended_categorical:
    print("\nCalculating Cardinality...")
    # Calculate for each column individually to avoid dtype issues in nunique()
    cardinality = {}
    for col in existing_intended_categorical:
        try:
            cardinality[col] = df[col].nunique()
        except Exception as e:
            print(f"Could not calculate nunique for {col}: {e}")
            cardinality[col] = -1 # Indicate error
    print("Cardinality (number of unique values):")
    for col, num in cardinality.items():
        print(f"  {col}: {num}")

else:
    print("\nNo intended categorical columns found.")

# Define columns for different encoding strategies based on INTENT and EXISTENCE
cols_for_freq_encoding = [col for col in ['ORIGIN', 'DEST'] if col in existing_intended_categorical]
# *** CRITICAL FIX: Base cols_for_ohe on existing_intended_categorical ***
cols_for_ohe = [col for col in ['AIRLINE', 'DEP_TIME_CATEGORY'] if col in existing_intended_categorical]

print(f"\nColumns assigned for Frequency Encoding: {cols_for_freq_encoding}")
print(f"Columns assigned for One-Hot Encoding: {cols_for_ohe}")

# Sanity check - ensure cols_for_ohe is NOT empty if expected
if not cols_for_ohe and ('AIRLINE' in df.columns or 'DEP_TIME_CATEGORY' in df.columns):
    print("\nWARNING: 'AIRLINE' or 'DEP_TIME_CATEGORY' exists but was not assigned to cols_for_ohe. Check logic.")

# --- End Replacement for Cell 6 ---

Columns PRESENT AT START of Cell 6: ['FL_DATE', 'AIRLINE', 'ORIGIN', 'DEST', 'DISTANCE', 'CRS_DEP_TIME', 'CRS_ARR_TIME', 'CRS_ELAPSED_TIME', 'IS_DELAYED', 'YEAR', 'MONTH', 'DAY', 'DAY_OF_WEEK', 'DAY_OF_YEAR', 'WEEK_OF_YEAR', 'DEP_HOUR', 'DEP_MINUTE', 'DEP_TIME_CATEGORY']
Identifying intended categorical features and checking cardinality...
Found intended categorical columns in df: ['AIRLINE', 'ORIGIN', 'DEST', 'DEP_TIME_CATEGORY']

Calculating Cardinality...
Cardinality (number of unique values):
  AIRLINE: 18
  ORIGIN: 380
  DEST: 380
  DEP_TIME_CATEGORY: 5

Columns assigned for Frequency Encoding: ['ORIGIN', 'DEST']
Columns assigned for One-Hot Encoding: ['AIRLINE', 'DEP_TIME_CATEGORY']


In [19]:
print("\nApplying Frequency Encoding...")

for col in cols_for_freq_encoding:
    # Calculate frequency map
    freq_map = df[col].value_counts(normalize=True) # Use normalize=True for percentage, False for count
    # Create new column name
    new_col_name = f"{col}_FREQ"
    # Map frequencies to the column
    df[new_col_name] = df[col].map(freq_map)
    print(f"Created '{new_col_name}' based on '{col}'.")
    # Optional: Drop the original column after encoding
    # df = df.drop(columns=[col])

print("Frequency Encoding applied.")

# Generate the flat list of columns to display
cols_to_display = []
for col in cols_for_freq_encoding:
    new_col_name = f"{col}_FREQ"
    if new_col_name in df.columns:
        cols_to_display.append(col)        # Add the original column
        cols_to_display.append(new_col_name) # Add the frequency column

# Check if there are any columns to display before trying to access them
if cols_to_display:
    print("Frequency Encoding applied.")
    display(df[cols_to_display].head())
else:
    print("Frequency Encoding applied, but no columns found to display (perhaps cols_for_freq_encoding was empty or columns weren't created).")



Applying Frequency Encoding...
Created 'ORIGIN_FREQ' based on 'ORIGIN'.
Created 'DEST_FREQ' based on 'DEST'.
Frequency Encoding applied.
Frequency Encoding applied.


,ORIGIN,ORIGIN_FREQ,DEST,DEST_FREQ
0,FLL,0.013416,EWR,0.017352
1,MSP,0.020209,SEA,0.023853
2,DEN,0.039934,MSP,0.020101
3,MSP,0.020209,SFO,0.019759
4,MCO,0.021287,DFW,0.042915


In [20]:
# --- Modify Cell 8 ---
print("\nApplying One-Hot Encoding...")

# --- Add these lines ---
print(f"Shape BEFORE OHE: {df.shape}")
print(f"Columns targeted for OHE: {cols_for_ohe}")
if 'AIRLINE' in cols_for_ohe:
    print(f"Unique values in AIRLINE: {df['AIRLINE'].nunique()}")
    print(f"Data type of AIRLINE: {df['AIRLINE'].dtype}")
    # print(f"Sample AIRLINE values: {df['AIRLINE'].unique()[:20]}") # Optional: see some unique values
# --- End Added lines ---

# Use pandas get_dummies for OHE
# dummy_na=False: Don't create a column for NaN values
# dtype=int: Create integer (0/1) columns
# --- Wrap in try-except ---
try:
    df_before_ohe_cols = set(df.columns) # Store columns before
    df = pd.get_dummies(df, columns=cols_for_ohe, dummy_na=False, dtype=int)
    print("One-Hot Encoding applied (pd.get_dummies finished).")
    df_after_ohe_cols = set(df.columns) # Store columns after
    new_ohe_cols = list(df_after_ohe_cols - df_before_ohe_cols)
    print(f"Total new OHE columns created (approx): {len(new_ohe_cols)}")
    # Check specifically for AIRLINE columns among the new ones
    new_airline_cols = [col for col in new_ohe_cols if col.startswith('AIRLINE_')]
    print(f"Number of new columns starting with 'AIRLINE_': {len(new_airline_cols)}")
    if not new_airline_cols and 'AIRLINE' in cols_for_ohe:
         print("WARNING: No 'AIRLINE_' columns were created by get_dummies!")

except Exception as e:
    print(f"ERROR during pd.get_dummies: {e}")
    print("OHE likely failed.")
# --- End Wrap ---


print(f"New shape AFTER OHE: {df.shape}")

# Display some of the new OHE columns
# --- Modified display ---
print("\nSample of new OHE columns (if any):")
ohe_cols_sample = df.filter(regex=f"^({'|'.join(cols_for_ohe)})_").columns[:15] # Show first 15 OHE cols overall
if not ohe_cols_sample.empty:
    display(df[ohe_cols_sample].head())
else:
    print("No columns found matching the OHE pattern for display.")
# --- End Modified display ---

# It's generally good practice to drop the original columns after Frequency or OHE
# We keep them for now for inspection, but will drop them before modeling.

# --- End Modify Cell 8 ---


Applying One-Hot Encoding...
Shape BEFORE OHE: (2913802, 20)
Columns targeted for OHE: ['AIRLINE', 'DEP_TIME_CATEGORY']
Unique values in AIRLINE: 18
Data type of AIRLINE: object
One-Hot Encoding applied (pd.get_dummies finished).
Total new OHE columns created (approx): 23
Number of new columns starting with 'AIRLINE_': 18
New shape AFTER OHE: (2913802, 41)

Sample of new OHE columns (if any):


,AIRLINE_Alaska Airlines Inc.,AIRLINE_Allegiant Air,AIRLINE_American Airlines Inc.,AIRLINE_Delta Air Lines Inc.,AIRLINE_Endeavor Air Inc.,AIRLINE_Envoy Air,AIRLINE_ExpressJet Airlines LLC d/b/a aha!,AIRLINE_Frontier Airlines Inc.,AIRLINE_Hawaiian Airlines Inc.,AIRLINE_Horizon Air,AIRLINE_JetBlue Airways,AIRLINE_Mesa Airlines Inc.,AIRLINE_PSA Airlines Inc.,AIRLINE_Republic Airline,AIRLINE_SkyWest Airlines Inc.
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [21]:
print("Finalizing feature set...")

# List original columns to drop
cols_to_drop = []

# Original categoricals that were frequency encoded
if 'ORIGIN' in df.columns and 'ORIGIN_FREQ' in df.columns:
    cols_to_drop.append('ORIGIN')
if 'DEST' in df.columns and 'DEST_FREQ' in df.columns:
    cols_to_drop.append('DEST')

# Original categoricals that were one-hot encoded (pandas get_dummies usually drops them, but check)
for col in cols_for_ohe: # ['OP_CARRIER', 'DEP_TIME_CATEGORY']
     if col in df.columns:
         # Check if OHE columns were actually created for it before dropping
         if any(df_col.startswith(f"{col}_") for df_col in df.columns):
              cols_to_drop.append(col)

# FL_DATE (we extracted YEAR, MONTH, DAY, etc.)
if 'FL_DATE' in df.columns:
    cols_to_drop.append('FL_DATE')

# CRS_DEP_TIME (we extracted DEP_HOUR, DEP_MINUTE, etc.)
if 'CRS_DEP_TIME' in df.columns:
     cols_to_drop.append('CRS_DEP_TIME')

# CRS_ARR_TIME (Scheduled arrival time - potentially useful, but let's drop for baseline)
# Models often focus on departure conditions + duration. We keep CRS_ELAPSED_TIME.
if 'CRS_ARR_TIME' in df.columns:
     cols_to_drop.append('CRS_ARR_TIME')

# Ensure columns actually exist before trying to drop
cols_to_drop = [col for col in cols_to_drop if col in df.columns]

print(f"Columns to drop: {cols_to_drop}")

df_final_features = df.drop(columns=cols_to_drop)

print("\nShape after dropping original/intermediate columns:", df_final_features.shape)
print("Final columns for modeling:")
print(df_final_features.columns.tolist())

# Final check for data types and NaNs
print("\nFinal data types:")
df_final_features.info()

print("\nFinal check for missing values:")
print(df_final_features.isnull().sum().sum()) # Should be 0

Finalizing feature set...
Columns to drop: ['ORIGIN', 'DEST', 'FL_DATE', 'CRS_DEP_TIME', 'CRS_ARR_TIME']

Shape after dropping original/intermediate columns: (2913802, 36)
Final columns for modeling:
['DISTANCE', 'CRS_ELAPSED_TIME', 'IS_DELAYED', 'YEAR', 'MONTH', 'DAY', 'DAY_OF_WEEK', 'DAY_OF_YEAR', 'WEEK_OF_YEAR', 'DEP_HOUR', 'DEP_MINUTE', 'ORIGIN_FREQ', 'DEST_FREQ', 'AIRLINE_Alaska Airlines Inc.', 'AIRLINE_Allegiant Air', 'AIRLINE_American Airlines Inc.', 'AIRLINE_Delta Air Lines Inc.', 'AIRLINE_Endeavor Air Inc.', 'AIRLINE_Envoy Air', 'AIRLINE_ExpressJet Airlines LLC d/b/a aha!', 'AIRLINE_Frontier Airlines Inc.', 'AIRLINE_Hawaiian Airlines Inc.', 'AIRLINE_Horizon Air', 'AIRLINE_JetBlue Airways', 'AIRLINE_Mesa Airlines Inc.', 'AIRLINE_PSA Airlines Inc.', 'AIRLINE_Republic Airline', 'AIRLINE_SkyWest Airlines Inc.', 'AIRLINE_Southwest Airlines Co.', 'AIRLINE_Spirit Air Lines', 'AIRLINE_United Air Lines Inc.', 'DEP_TIME_CATEGORY_Night', 'DEP_TIME_CATEGORY_Morning', 'DEP_TIME_CATEGORY_Af

In [22]:
# Define path for the final feature-engineered data
output_dir = '../data'
# Make sure the directory exists (it should from the previous notebook)
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created directory: {output_dir}")

# Define the output filename
output_file_features = os.path.join(output_dir, 'flights_features_v1.csv')

print(f"\nSaving final feature-engineered data to: {output_file_features} ...")
# Use index=False to avoid writing the dataframe index as a column
df_final_features.to_csv(output_file_features, index=False)

print("Feature-engineered data saved successfully.")
print(f"Final shape saved: {df_final_features.shape}")

# Optional: Verify file size
file_size_mb = os.path.getsize(output_file_features) / (1024 * 1024)
print(f"Saved file size: {file_size_mb:.2f} MB")



Saving final feature-engineered data to: ../data\flights_features_v1.csv ...
Feature-engineered data saved successfully.
Final shape saved: (2913802, 36)
Saved file size: 351.69 MB
